# Tratamento dos dados — StockFlow (Bronze → Silver)

Sprint 2. Segue o **Plano de Preparação de Dados** (seções 3, 4 e 6).

- Entrada (Bronze): `data/bronze/sales_data.csv`, dataset *Retail Store Inventory and Demand Forecasting* (Kaggle). Nunca é alterado.
- Saída (Silver): `data/silver/vendas_tratadas.parquet`, com tipos corrigidos, duplicados e erros removidos, faltantes tratados conforme o mecanismo, categorias padronizadas e extremos marcados.
- Registro: `docs/log_tratamento.md`.

Variáveis derivadas, rótulo de ruptura, codificação, padronização e divisão treino/teste ficam para a camada **Gold** (Sprint 3 em diante), como definido na seção 6 do plano.

## 1. Importações e leitura do CSV

In [1]:
import os
import subprocess
import pandas as pd

CSV = "data/bronze/sales_data.csv"

# Funciona no Colab e localmente: se o CSV não estiver acessível, clona o repositório
if not os.path.exists(CSV):
    if os.path.exists(os.path.join("..", CSV)):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "https://github.com/juliamariateixeiraa/StockFlow.git"], check=True)
        os.chdir("StockFlow")

df_bruto = pd.read_csv(CSV)
df = df_bruto.copy()
print(df.shape)

(76000, 16)


## 2. Visão geral dos dados

Confirma o que o plano deixou para a Sprint 2: a data final do período e a lista completa de classes de clima e das outras categóricas.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  str    
 1   Store ID            76000 non-null  str    
 2   Product ID          76000 non-null  str    
 3   Category            76000 non-null  str    
 4   Region              76000 non-null  str    
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  str    
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  str    
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: float64(2), 

In [3]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Inventory Level,76000.0,301.062842,226.510161,0.00,136.0000,227.0,408.0000,2267.00
Units Sold,76000.0,88.827316,43.994525,0.00,58.0000,84.0,114.0000,426.00
Units Ordered,76000.0,89.090645,162.404627,0.00,0.0000,0.0,121.0000,1616.00
Price,76000.0,67.726028,39.377899,4.74,31.9975,64.5,95.8300,228.03
Discount,76000.0,9.087039,7.475781,0.00,5.0000,10.0,10.0000,25.00
Promotion,76000.0,0.328947,0.469834,0.00,0.0000,0.0,1.0000,1.00
Competitor Pricing,76000.0,69.454029,40.943818,4.29,32.6200,65.7,97.9325,261.22
Epidemic,76000.0,0.200000,0.400003,0.00,0.0000,0.0,0.0000,1.00
Demand,76000.0,104.317158,46.964801,4.00,71.0000,100.0,133.0000,430.00


In [5]:
print("Período:", df["Date"].min(), "a", df["Date"].max())
print("Dias distintos:", df["Date"].nunique())
print("Lojas:", df["Store ID"].nunique(), "| Produtos:", df["Product ID"].nunique())

for col in ["Category", "Region", "Weather Condition", "Seasonality"]:
    print(f"{col}: {sorted(df[col].unique())}")
for col in ["Promotion", "Epidemic"]:
    print(f"{col}: {sorted(df[col].unique())}")
print("Discount (degraus):", sorted(df["Discount"].unique()))

Período: 2022-01-01 a 2024-01-30
Dias distintos: 760
Lojas: 5 | Produtos: 20
Category: ['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']
Region: ['East', 'North', 'South', 'West']
Weather Condition: ['Cloudy', 'Rainy', 'Snowy', 'Sunny']
Seasonality: ['Autumn', 'Spring', 'Summer', 'Winter']
Promotion: [np.int64(0), np.int64(1)]
Epidemic: [np.int64(0), np.int64(1)]
Discount (degraus): [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25)]


## 3. Padronização dos nomes das colunas e do texto

- Nomes das colunas em minúsculas e sem espaços (snake_case), para facilitar o uso no código e no banco.
- Texto das categóricas: retira espaços extras e padroniza maiúsculas, para que "Rainy" e "rainy " não virem classes diferentes (seção 4 do plano).

In [6]:
nomes = {
    "Date": "data",
    "Store ID": "loja_id",
    "Product ID": "produto_id",
    "Category": "categoria",
    "Region": "regiao",
    "Inventory Level": "estoque",
    "Units Sold": "unidades_vendidas",
    "Units Ordered": "unidades_pedidas",
    "Price": "preco",
    "Discount": "desconto",
    "Weather Condition": "clima",
    "Promotion": "promocao",
    "Competitor Pricing": "preco_concorrente",
    "Seasonality": "estacao",
    "Epidemic": "epidemia",
    "Demand": "demanda",
}
df = df.rename(columns=nomes)

texto = ["loja_id", "produto_id", "categoria", "regiao", "clima", "estacao"]
for col in texto:
    antes = df[col].nunique()
    df[col] = df[col].astype(str).str.strip()
    if col not in ["loja_id", "produto_id"]:
        df[col] = df[col].str.title()
    depois = df[col].nunique()
    print(f"{col}: {antes} classes antes, {depois} depois")

loja_id: 5 classes antes, 5 depois
produto_id: 20 classes antes, 20 depois
categoria: 5 classes antes, 5 depois
regiao: 4 classes antes, 4 depois
clima: 4 classes antes, 4 depois
estacao: 4 classes antes, 4 depois


## 4. Conversão de tipos

Conforme a seção 4 do plano:
- `data` como data;
- IDs e categóricas como texto;
- quantidades como inteiro;
- `promocao` e `epidemia` continuam 0/1, já prontas para o modelo (seção 2).

In [7]:
df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d")

for col in texto:
    df[col] = df[col].astype("string")

for col in ["estoque", "unidades_vendidas", "unidades_pedidas", "desconto", "demanda", "promocao", "epidemia"]:
    df[col] = df[col].astype("int64")

for col in ["preco", "preco_concorrente"]:
    df[col] = df[col].astype("float64")

df.dtypes

data                 datetime64[us]
loja_id                      string
produto_id                   string
categoria                    string
regiao                       string
estoque                       int64
unidades_vendidas             int64
unidades_pedidas              int64
preco                       float64
desconto                      int64
clima                        string
promocao                      int64
preco_concorrente           float64
estacao                      string
epidemia                      int64
demanda                       int64
dtype: object

## 5. Duplicados

A chave **data × loja × produto** deve ser única. Se houver repetição, mantém só uma linha.

In [8]:
chave = ["data", "loja_id", "produto_id"]
print("Linhas idênticas:", df.duplicated().sum())
print("Chave data × loja × produto repetida:", df.duplicated(subset=chave).sum())

linhas_antes = len(df)
df = df.drop_duplicates(subset=chave, keep="first")
print("Linhas removidas:", linhas_antes - len(df))

Linhas idênticas: 0
Chave data × loja × produto repetida: 0
Linhas removidas: 0


## 6. Dados faltantes e mecanismo de falta

Seção 3 do plano: antes de limpar, verificar o que falta e por que falta.

### Passo 1 — Verificar o que falta

**a) Nulos por coluna**

In [9]:
nulos = df.isna().sum()
print(nulos[nulos > 0] if nulos.sum() else "Nenhum valor nulo.")

Nenhum valor nulo.


**b) Linhas ausentes**

Toda combinação data × loja × produto deveria existir. Uma linha que não existe também é dado faltante.

In [10]:
datas = pd.date_range(df["data"].min(), df["data"].max(), freq="D")
completo = pd.MultiIndex.from_product(
    [datas, sorted(df["loja_id"].unique()), sorted(df["produto_id"].unique())], names=chave
)
existentes = pd.MultiIndex.from_frame(df[chave])
faltando = completo.difference(existentes)

print("Combinações esperadas:", len(completo))
print("Combinações existentes:", len(existentes))
print("Linhas ausentes:", len(faltando))
print("Dias faltando no calendário:", len(datas) - df["data"].nunique())

Combinações esperadas: 76000
Combinações existentes: 76000
Linhas ausentes: 0
Dias faltando no calendário: 0


**c) Zeros e valores impossíveis que podem esconder um faltante**

`unidades_pedidas = 0` significa que não houve pedido; não é dado faltante (seção 2 do plano).

In [11]:
zeros = pd.DataFrame({
    "zeros": [(df[c] == 0).sum() for c in ["estoque", "unidades_vendidas", "unidades_pedidas", "demanda", "preco", "preco_concorrente"]],
}, index=["estoque", "unidades_vendidas", "unidades_pedidas", "demanda", "preco", "preco_concorrente"])
zeros["percentual"] = (zeros["zeros"] / len(df) * 100).round(2)
zeros

,zeros,percentual
estoque,406,0.53
unidades_vendidas,406,0.53
unidades_pedidas,47107,61.98
demanda,0,0.00
preco,0,0.00
preco_concorrente,0,0.00


In [12]:
# Estoque zerado: a loja não tinha o produto. Venda zero nesses dias é consequência, não falta de registro.
estoque_zero = df[df["estoque"] == 0]
print("Dias com estoque zero:", len(estoque_zero))
print("Desses, com venda zero:", (estoque_zero["unidades_vendidas"] == 0).sum())
print("Dias com venda zero e estoque maior que zero:", ((df["unidades_vendidas"] == 0) & (df["estoque"] > 0)).sum())

Dias com estoque zero: 406
Desses, com venda zero: 406
Dias com venda zero e estoque maior que zero: 0


### Passo 2 — Classificar o mecanismo

Se houvesse nulos, a taxa de nulos seria comparada entre lojas, categorias e meses para separar MCAR, MAR e MNAR. A regra fica pronta para quando a importação de planilhas trouxer dados reais incompletos.

In [13]:
def taxa_nulos_por_grupo(base, grupo):
    return base.isna().any(axis=1).groupby(grupo).mean().mul(100).round(2)

if df.isna().sum().sum() == 0:
    print("Sem nulos: não há mecanismo de falta a classificar nesta base.")
else:
    for g in ["loja_id", "categoria", df["data"].dt.month.rename("mes")]:
        print(taxa_nulos_por_grupo(df, g), "\n")

    # Tratamento previsto no plano:
    # MCAR -> remover a linha, se for pouca coisa
    # MAR  -> preencher com a mediana do mesmo produto na mesma loja
    # MNAR -> não preencher às cegas; marcar com coluna indicadora

Sem nulos: não há mecanismo de falta a classificar nesta base.


### Passo 3 — Venda censurada (MNAR invisível)

Quando o estoque acaba, a loja só vende o que tinha e a demanda que sobrou não aparece em `unidades_vendidas`. O plano prevê usar `demanda` como alvo da regressão, **se a coluna estiver disponível e consistente**. Por isso a consistência dela é verificada aqui.

In [14]:
venda_igual_estoque = (df["unidades_vendidas"] == df["estoque"])
demanda_maior_estoque = (df["demanda"] > df["estoque"])

print("Venda = estoque (vendeu tudo):", venda_igual_estoque.sum(), f"({venda_igual_estoque.mean():.1%})")
print("Demanda > estoque:", demanda_maior_estoque.sum(), f"({demanda_maior_estoque.mean():.1%})")
print("Venda ≥ estoque OU demanda > estoque:", (venda_igual_estoque | demanda_maior_estoque).sum(),
      f"({(venda_igual_estoque | demanda_maior_estoque).mean():.1%})")

Venda = estoque (vendeu tudo): 8654 (11.4%)
Demanda > estoque: 10394 (13.7%)
Venda ≥ estoque OU demanda > estoque: 11200 (14.7%)


**Consistência de `demanda`**

Se `demanda` é o que o cliente quis comprar, a venda nunca deveria passar dela.

In [15]:
venda_maior_demanda = df["unidades_vendidas"] > df["demanda"]
dif = (df["unidades_vendidas"] - df["demanda"])[venda_maior_demanda]

print("Venda > demanda:", venda_maior_demanda.sum(), f"({venda_maior_demanda.mean():.1%} das linhas)")
print(dif.describe().round(1))

Venda > demanda: 21000 (27.6% das linhas)
count    21000.0
mean        10.1
std          7.5
min          1.0
25%          4.0
50%          8.0
75%         14.0
max         68.0
dtype: float64


In [16]:
# A inconsistência se concentra em algum grupo?
pd.concat({
    "por loja (%)": venda_maior_demanda.groupby(df["loja_id"]).mean().mul(100).round(1),
}, axis=1)

,por loja (%)
loja_id,
S001,27.7
S002,28.2
S003,26.9
S004,28.3
S005,27.0


In [17]:
pd.concat({
    "por categoria (%)": venda_maior_demanda.groupby(df["categoria"]).mean().mul(100).round(1),
}, axis=1)

,por categoria (%)
categoria,
Clothing,25.7
Electronics,27.2
Furniture,30.2
Groceries,27.8
Toys,26.5


In [18]:
pd.concat({
    "promocao (%)": venda_maior_demanda.groupby(df["promocao"]).mean().mul(100).round(1),
}, axis=1)

,promocao (%)
promocao,
0,28.2
1,26.4


As linhas com venda maior que a demanda **não são removidas**: não dá para saber qual dos dois valores está errado. Elas são marcadas com a coluna `flag_venda_maior_demanda`.

Como o plano condiciona o uso de `demanda` à consistência da coluna, essa marcação define uma decisão para o grupo antes da regressão (Sprint 4): usar `demanda` mesmo assim ou seguir a alternativa do plano (treinar sem os dias de ruptura ou com marcador e comparar).

In [19]:
df["flag_venda_maior_demanda"] = venda_maior_demanda.astype(int)

## 7. Valores impossíveis, venda acima do estoque e consistência entre colunas

Seção 4 do plano:
- estoque, venda ou preço negativos, preço zero e desconto acima de 100 são erros: corrigir se for óbvio, senão remover e registrar;
- venda acima do estoque deve ser investigada antes de agir, porque pode indicar reposição no mesmo dia.
- consistência entre colunas: só verificação, nenhum dado é alterado.

In [20]:
regras = {
    "estoque negativo": df["estoque"] < 0,
    "venda negativa": df["unidades_vendidas"] < 0,
    "pedido negativo": df["unidades_pedidas"] < 0,
    "demanda negativa": df["demanda"] < 0,
    "preço ≤ 0": df["preco"] <= 0,
    "preço do concorrente ≤ 0": df["preco_concorrente"] <= 0,
    "desconto fora de 0–100": ~df["desconto"].between(0, 100),
}
for nome, mask in regras.items():
    print(f"{nome}: {mask.sum()}")

impossiveis = pd.concat(regras, axis=1).any(axis=1)
linhas_antes = len(df)
df = df[~impossiveis].copy()
print("Linhas removidas:", linhas_antes - len(df))

estoque negativo: 0
venda negativa: 0
pedido negativo: 0
demanda negativa: 0
preço ≤ 0: 0
preço do concorrente ≤ 0: 0
desconto fora de 0–100: 0
Linhas removidas: 0


In [21]:
venda_acima_estoque = df["unidades_vendidas"] > df["estoque"]
print("Venda > estoque:", venda_acima_estoque.sum())

Venda > estoque: 0


Não há venda acima do estoque. Isso indica que o estoque registrado já é o disponível para venda no dia, e não o que sobrou no fim do dia. Esse ponto ajuda a fechar a regra do rótulo de ruptura na Gold, que o plano deixou dependente de confirmar se o estoque é do início ou do fim do dia.

In [22]:
consistencia = {
    "categorias por loja × produto": df.groupby(["loja_id", "produto_id"])["categoria"].nunique().max(),
    "categorias por produto_id (todas as lojas)": df.groupby("produto_id")["categoria"].nunique().max(),
    "regiões por loja": df.groupby("loja_id")["regiao"].nunique().max(),
    "climas por dia × região": df.groupby(["data", "regiao"])["clima"].nunique().max(),
}
for nome, valor in consistencia.items():
    print(f"{nome}: {valor}")

categorias por loja × produto: 1
categorias por produto_id (todas as lojas): 5
regiões por loja: 1
climas por dia × região: 1


In [23]:
# Estação x mês (hemisfério norte: dez–fev inverno, mar–mai primavera, jun–ago verão, set–nov outono)
estacao_esperada = df["data"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
})
print("Estação diferente do mês:", (df["estacao"] != estacao_esperada).sum())

# Promoção x desconto
pd.crosstab(df["promocao"], df["desconto"])

Estação diferente do mês: 0


desconto,0,5,10,15,20,25
promocao,,,,,,
0,17126,16918,16956,0,0,0
1,0,0,6342,6245,6191,6222


**Consistência entre colunas**

- Categoria: é fixa em cada par loja × produto, mas o mesmo `produto_id` aparece com categoria diferente em cada loja. Por isso a série de cada item é loja × produto, e é assim que o IQR (etapa 8) e as defasagens e médias móveis (Gold) são calculados.
- Região: cada loja tem uma só região.
- Clima: é o mesmo para todas as lojas de uma região no mesmo dia.
- Estação: bate com o mês em todas as linhas.
- Promoção e desconto: com promoção, o desconto vai de 10 a 25; sem promoção, de 0 a 10. Não há conflito.

Nenhum dado foi alterado nesta verificação.

## 8. Valores extremos

Regra do IQR **por loja × produto** (não na base toda). O plano fala em IQR por produto, mas a verificação de consistência da etapa 7 mostrou que o mesmo `produto_id` tem categoria diferente em cada loja (o P0001, por exemplo, é de uma categoria na S001 e de outra nas demais lojas). Ou seja, o item de verdade é o par loja × produto, e é por ele que o IQR é calculado: cada item tem seu próprio nível de venda, e um valor alto para um item pode ser normal para outro. Os extremos são **marcados, não removidos**, porque picos em promoção ou feriado são o que o modelo precisa aprender.

`unidades_pedidas` fica fora: a maior parte dos valores é zero (dias sem pedido), o que distorce o IQR.

In [24]:
print("Zeros em unidades_pedidas:", f"{(df['unidades_pedidas'] == 0).mean():.1%}")

def marcar_extremos(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return ((serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)).astype(int)

resumo_extremos = []
for col in ["unidades_vendidas", "demanda", "estoque", "preco"]:
    df[f"extremo_{col}"] = df.groupby(["loja_id", "produto_id"])[col].transform(marcar_extremos)
    resumo_extremos.append({
        "coluna": col,
        "extremos": int(df[f"extremo_{col}"].sum()),
        "percentual": f"{df[f'extremo_{col}'].mean():.1%}",
    })
pd.DataFrame(resumo_extremos)

Zeros em unidades_pedidas: 62.0%


,coluna,extremos,percentual
0,unidades_vendidas,1104,1.5%
1,demanda,804,1.1%
2,estoque,1119,1.5%
3,preco,550,0.7%


In [25]:
# Os extremos de venda acontecem mais em promoção?
pd.DataFrame({
    "% de extremos de venda": df.groupby("promocao")["extremo_unidades_vendidas"].mean().mul(100).round(1)
})

,% de extremos de venda
promocao,
0,0.5
1,3.5


## 9. Período de epidemia

O plano decide manter `epidemia` como variável e testar o modelo com e sem esse período. Aqui só é registrado o tamanho e a localização do período.

In [26]:
epi = df[df["epidemia"] == 1]
print("Linhas com epidemia:", len(epi), f"({len(epi) / len(df):.1%})")
print("Dias com epidemia:", epi["data"].nunique())
print("Primeiro e último dia:", epi["data"].min().date(), "a", epi["data"].max().date())
print("Datas mistas (com e sem epidemia no mesmo dia):",
      (df.groupby("data")["epidemia"].nunique() > 1).sum())

Linhas com epidemia: 15200 (20.0%)
Dias com epidemia: 152
Primeiro e último dia: 2022-04-23 a 2024-01-08
Datas mistas (com e sem epidemia no mesmo dia): 0


In [27]:
df.groupby("epidemia")[["unidades_vendidas", "demanda"]].mean().round(1)

,unidades_vendidas,demanda
epidemia,,
0,95.8,112.9
1,60.9,70.2


## 10. Resumo antes x depois

In [28]:
resumo = pd.DataFrame({
    "Bronze (original)": [len(df_bruto), df_bruto.shape[1], int(df_bruto.isna().sum().sum()), int(df_bruto.duplicated(subset=["Date", "Store ID", "Product ID"]).sum())],
    "Silver (tratada)": [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated(subset=chave).sum())],
}, index=["Linhas", "Colunas", "Valores nulos", "Chave data × loja × produto repetida"])
resumo

,Bronze (original),Silver (tratada)
Linhas,76000,76000
Colunas,16,21
Valores nulos,0,0
Chave data × loja × produto repetida,0,0


In [29]:
print("Colunas de marcação adicionadas:", [c for c in df.columns if c not in nomes.values()])

Colunas de marcação adicionadas: ['flag_venda_maior_demanda', 'extremo_unidades_vendidas', 'extremo_demanda', 'extremo_estoque', 'extremo_preco']


## 11. Salvar em `data/silver/vendas_tratadas.parquet`

Parquet guarda os tipos (data, texto, inteiro) e evita refazer a conversão nos próximos notebooks (seção 6 do plano). A carga no banco de dados acontece quando a aplicação for integrada (Sprints 7 e 8).

In [30]:
df = df.sort_values(chave).reset_index(drop=True)
os.makedirs("data/silver", exist_ok=True)
df.to_parquet("data/silver/vendas_tratadas.parquet", index=False)

conferencia = pd.read_parquet("data/silver/vendas_tratadas.parquet")
print("Salvo:", conferencia.shape)
conferencia.dtypes

Salvo: (76000, 21)


data                         datetime64[us]
loja_id                              string
produto_id                           string
categoria                            string
regiao                               string
estoque                               int64
unidades_vendidas                     int64
unidades_pedidas                      int64
preco                               float64
desconto                              int64
clima                                string
promocao                              int64
preco_concorrente                   float64
estacao                              string
epidemia                              int64
demanda                               int64
flag_venda_maior_demanda              int64
extremo_unidades_vendidas             int64
extremo_demanda                       int64
extremo_estoque                       int64
extremo_preco                         int64
dtype: object